# Arohan Medical First Aid — Full Fine-Tuning Pipeline

**Pipeline:** SFT Dataset → QLoRA (SFT) → DPO → Evaluation

This notebook orchestrates all three workstreams:
- **Vaibhav:** SFT dataset (1,040 train / 116 val pairs)
- **Amulya:** QLoRA PEFT fine-tuning (3 experiments: baseline/medium/high)
- **Anisha:** DPO preference alignment (47 preference pairs)

---
## Setup Instructions

**Before running this notebook:**
1. Upload the `sft_data/` folder to `MyDrive/arohan_sft_data/` in your Google Drive
2. Ensure you have a free T4 GPU (Runtime -> Change runtime type -> T4 GPU)

**Estimated time:** 20-30 minutes total (SFT: ~15 min, DPO: ~5 min, Eval: ~5 min)

In [ ]:
# Cell 1: Check GPU & Environment
import torch
import sys
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', None))
    if vram:
        print(f"VRAM: {vram / 1024**3:.1f} GB")
    else:
        print("VRAM: unable to detect")

assert torch.cuda.is_available(), "GPU required! Go to Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# Cell 2: Install Dependencies (~2 min)
print("Installing Unsloth (fast 4-bit QLoRA)...")
!pip install -q unsloth

print("Installing TRL, PEFT, bitsandbytes...")
!pip install -q --no-deps trl peft accelerate bitsandbytes

print("Installing datasets, groq...")
!pip install -q datasets groq

print("\nVerifying installations...")
import unsloth
from unsloth import FastLanguageModel
import trl
import peft
import datasets
print(f"unsloth: {unsloth.__version__ if hasattr(unsloth, '__version__') else 'ok'}")
print(f"trl: {trl.__version__}")
print(f"peft: {peft.__version__}")
print(f"datasets: {datasets.__version__}")
print("\nAll dependencies installed!")

In [ ]:
# Cell 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify data exists
import os
DATA_DIR = "/content/drive/MyDrive/arohan_sft_data"
required_files = [
    "sft_dataset_train.jsonl",
    "sft_dataset_val.jsonl",
    "dpo_dataset_train.jsonl",
    "dpo_dataset_val.jsonl",
]

print(f"\nChecking data in {DATA_DIR}:")
for f in required_files:
    path = os.path.join(DATA_DIR, f)
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"  [OK]   {f:40s} {size:>8,} bytes")
    else:
        print(f"  [MISS] {f:40s}")

all_ok = all(os.path.exists(os.path.join(DATA_DIR, f)) for f in required_files)
assert all_ok, "Missing files! Upload sft_data/ to MyDrive/arohan_sft_data/"
print("\nAll dataset files found! Ready to train.")

In [ ]:
# Cell 4: Copy scripts & data to Colab runtime for faster access
import shutil
import os

# Create working directory
WORK_DIR = "/content/arohan_pipeline"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {WORK_DIR}")

# Copy data from Drive to local runtime (faster I/O during training)
DATA_SRC = "/content/drive/MyDrive/arohan_sft_data"
DATA_DST = os.path.join(WORK_DIR, "sft_data")

if not os.path.exists(DATA_DST):
    print("Copying data from Drive to local runtime...")
    shutil.copytree(DATA_SRC, DATA_DST)
    print(f"  Copied to {DATA_DST}")

# Verify SFT data
import json
with open(os.path.join(DATA_DST, "sft_dataset_train.jsonl"), "r") as f:
    first = json.loads(f.readline())
print(f"\nSFT sample keys: {list(first.keys())}")
print(f"Has 'text' field: {'text' in first}")
print(f"Llama 3 template: {'<|begin_of_text|>' in first['text']}")

# Verify DPO data
with open(os.path.join(DATA_DST, "dpo_dataset_train.jsonl"), "r") as f:
    first = json.loads(f.readline())
print(f"DPO sample keys: {list(first.keys())}")
print(f"Has prompt/chosen/rejected: {all(k in first for k in ['prompt','chosen','rejected'])}")

# Count pairs
train_count = sum(1 for _ in open(os.path.join(DATA_DST, "sft_dataset_train.jsonl")))
val_count = sum(1 for _ in open(os.path.join(DATA_DST, "sft_dataset_val.jsonl")))
dpo_train_count = sum(1 for _ in open(os.path.join(DATA_DST, "dpo_dataset_train.jsonl")))
dpo_val_count = sum(1 for _ in open(os.path.join(DATA_DST, "dpo_dataset_val.jsonl")))

print(f"\nDataset summary:")
print(f"  SFT train:  {train_count} pairs")
print(f"  SFT val:    {val_count} pairs")
print(f"  DPO train:  {dpo_train_count} pairs")
print(f"  DPO val:    {dpo_val_count} pairs")
print(f"\nAll data verified! Ready to train.")

---
## Run the Full Pipeline

The following cells run all three stages sequentially:

1. **QLoRA (SFT)** — Trains LoRA adapters on the 1,040 first aid Q&A pairs
2. **DPO** — Applies preference alignment on top of the QLoRA adapter
3. **Evaluation** — Compares the trained models against Groq production

Each stage runs in a separate subprocess so GPU memory is freed between stages.

In [ ]:
# Cell 5: Stage 1 — QLoRA SFT Training (Amulya's PEFT)
# ~10-15 minutes on T4 GPU

print("=" * 70)
print("STAGE 1: QLoRA SFT Training")
print("=" * 70)

# Run QLoRA training as a subprocess so GPU memory is freed after stage completes
!cd "{WORK_DIR}" && python -c "
import os
os.chdir('/content/arohan_pipeline')

# Patch paths for Colab
import step4_qlora_train as sft
sft.DATASET_DIR = '/content/arohan_pipeline/sft_data'
sft.TRAIN_FILE = os.path.join(sft.DATASET_DIR, 'sft_dataset_train.jsonl')
sft.VAL_FILE = os.path.join(sft.DATASET_DIR, 'sft_dataset_val.jsonl')
sft.OUTPUT_DIR = '/content/arohan_pipeline/arohan_qlora_adapter'
sft.ACTIVE_EXPERIMENT = 'medium'

# Run training
model, tokenizer = sft.load_base_model()
model = sft.add_lora_adapters(model, sft.EXPERIMENTS[sft.ACTIVE_EXPERIMENT])
train_dataset, val_dataset = sft.load_datasets()
sft.train(model, tokenizer, train_dataset, val_dataset, sft.EXPERIMENTS[sft.ACTIVE_EXPERIMENT])
sft.test_inference(model, tokenizer)
" 2>&1

# Copy adapter to Drive for persistence
DRIVE_QLORA = "/content/drive/MyDrive/arohan_qlora_adapter"
if not os.path.exists(DRIVE_QLORA):
    shutil.copytree(os.path.join(WORK_DIR, "arohan_qlora_adapter"), DRIVE_QLORA)
    print(f"Adapter backed up to Google Drive: {DRIVE_QLORA}")

print("\n" + "=" * 70)
print("STAGE 1 COMPLETE — QLoRA adapter saved!")
print("=" * 70)

In [ ]:
# Cell 5b: Copy pipeline Python scripts to Colab runtime for import
import os, shutil

expected_scripts = [
    "step4_qlora_train.py",
    "step4c_dpo_train.py",
    "step5_evaluate_comprehensive.py",
    "inference_qlora.py",
    "pipeline_full_finetune.py",
]

# Scripts should already be in WORK_DIR (part of upload), but also try Drive backup
drive_script_dir = "/content/drive/MyDrive/arohan_scripts"
if os.path.exists(drive_script_dir):
    for script in expected_scripts:
        src = os.path.join(drive_script_dir, script)
        dst = os.path.join(WORK_DIR, script)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy2(src, dst)
            print(f"  Copied {script} from Drive")

# Verify all scripts are available
for script in expected_scripts:
    local_path = os.path.join(WORK_DIR, script)
    if os.path.exists(local_path):
        print(f"  [OK] {script}")
    else:
        print(f"  [MISS] {script} - will look in current directory")

# Add WORK_DIR to Python path so imports work from subprocesses
import sys
sys.path.insert(0, WORK_DIR)
print(f"\nAdded {WORK_DIR} to Python path")

In [ ]:
# Cell 6: Stage 2 — DPO Preference Alignment (Anisha's Work)
# ~5 minutes on T4 GPU

print("=" * 70)
print("STAGE 2: DPO Preference Alignment")
print("=" * 70)

# Auto-detect the experiment folder from QLoRA output
import os
qlora_exp_dir = "/content/arohan_pipeline/arohan_qlora_adapter"
if os.path.exists(qlora_exp_dir):
    exp_folders = sorted([d for d in os.listdir(qlora_exp_dir) if d.startswith("experiment_")])
    if exp_folders:
        QLORA_ADAPTER_PATH = os.path.join(qlora_exp_dir, exp_folders[-1])
        print(f"Auto-detected QLoRA adapter: {exp_folders[-1]}")
    else:
        print("No experiment folders found, using default")
        QLORA_ADAPTER_PATH = "/content/arohan_pipeline/arohan_qlora_adapter/experiment_medium"
else:
    QLORA_ADAPTER_PATH = "/content/arohan_pipeline/arohan_qlora_adapter/experiment_medium"

print(f"Adapter path: {QLORA_ADAPTER_PATH}")

if not os.path.exists(QLORA_ADAPTER_PATH):
    print(f"WARNING: QLoRA adapter not found! Check Cell 5 ran successfully.")

# Pass path to subprocess via environment variable
os.environ["AROHAN_QLORA_ADAPTER"] = QLORA_ADAPTER_PATH

!cd "{WORK_DIR}" && python -c "
import os
os.chdir('/content/arohan_pipeline')

# Read adapter path from env var set by parent process
adapter_path = os.environ.get('AROHAN_QLORA_ADAPTER', '/content/arohan_pipeline/arohan_qlora_adapter/experiment_medium')

import step4c_dpo_train as dpo
dpo.DATASET_DIR = '/content/arohan_pipeline/sft_data'
dpo.DPO_TRAIN_FILE = os.path.join(dpo.DATASET_DIR, 'dpo_dataset_train.jsonl')
dpo.DPO_VAL_FILE = os.path.join(dpo.DATASET_DIR, 'dpo_dataset_val.jsonl')
dpo.QLORA_ADAPTER_PATH = adapter_path
dpo.OUTPUT_DIR = '/content/arohan_pipeline/arohan_dpo_adapter'

model, tokenizer = dpo.load_pretrained_qlora()
train_dataset, val_dataset = dpo.load_dpo_datasets()
trainer, trainer_stats = dpo.run_dpo_training(model, tokenizer, train_dataset, val_dataset)
dpo.save_dpo_adapter(model, tokenizer, train_dataset, val_dataset)
" 2>&1

# Copy DPO adapter to Drive
DRIVE_DPO = "/content/drive/MyDrive/arohan_dpo_adapter"
if not os.path.exists(DRIVE_DPO):
    shutil.copytree(os.path.join(WORK_DIR, "arohan_dpo_adapter"), DRIVE_DPO)
    print(f"DPO adapter backed up to Google Drive: {DRIVE_DPO}")

print("\n" + "=" * 70)
print("STAGE 2 COMPLETE — DPO adapter saved!")
print("=" * 70)

In [ ]:
# Cell 7: Stage 3 — Comprehensive Evaluation
# ~5 minutes (loads each model and runs 20 test cases)

print("=" * 70)
print("STAGE 3: Comprehensive Evaluation")
print("=" * 70)

# Check available models
import os
from types import SimpleNamespace

QLORA_PATH = os.path.join(WORK_DIR, "arohan_qlora_adapter", "experiment_medium")
DPO_PATH = os.path.join(WORK_DIR, "arohan_dpo_adapter")

has_qlora = os.path.exists(QLORA_PATH)
has_dpo = os.path.exists(DPO_PATH)
groq_key = os.getenv("GROQ_API_KEY")

print(f"QLoRa adapter found: {has_qlora} at {QLORA_PATH}")
print(f"DPO adapter found:   {has_dpo} at {DPO_PATH}")
print(f"GROQ_API_KEY set:    {bool(groq_key)}")

# Build args using SimpleNamespace instead of argparse
eval_args = SimpleNamespace(
    groq=bool(groq_key),
    qlora_adapter=QLORA_PATH if has_qlora else None,
    dpo_adapter=DPO_PATH if has_dpo else None,
    all=False,
    offline=False,
)

# Add WORK_DIR to Python path for importing step5
import sys
sys.path.insert(0, WORK_DIR)

from step5_evaluate_comprehensive import evaluate_all_models
from step5_evaluate_comprehensive import OUTPUT_DIR as EVAL_OUTPUT_DIR

results, summary = evaluate_all_models(eval_args)

print("\n" + "=" * 70)
print("EVALUATION COMPLETE!")
print("=" * 70)

# Copy eval report to Drive
EVAL_DST = "/content/drive/MyDrive/arohan_eval_results"
os.makedirs(EVAL_DST, exist_ok=True)
eval_report = os.path.join(EVAL_OUTPUT_DIR, "comprehensive_eval_report.json")
if os.path.exists(eval_report):
    import shutil
    shutil.copy(eval_report, EVAL_DST)
    print(f"Evaluation report backed up to: {EVAL_DST}")

print("\n" + "=" * 70)
print("ALL STAGES COMPLETE!")
print("=" * 70)

In [ ]:
# Cell 8: Display Results
import json
import os

report_path = "/content/evals/comprehensive_eval_report.json"
if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        report = json.load(f)

    summary = report.get("summary", {})

    print("=" * 80)
    print("  FINAL EVALUATION REPORT")
    print("  Arohan Model Comparison")
    print("=" * 80)

    if summary:
        print(f"\n{'Model':<35} {'Score':<10} {'Steps':<8} {'112/Emerg':<12} {'Latency':<10}")
        print("-" * 75)
        for model_name, stats in sorted(summary.items(), key=lambda x: -x[1]["avg_overall_score"]):
            score = stats["avg_overall_score"]
            steps = stats["avg_steps_per_response"]
            emerg = stats["has_112_in_emergencies"]
            lat = stats["avg_latency_seconds"]
            print(f"{model_name:<35} {score:<10.3f} {steps:<8.1f} {emerg:<12} {lat:<10.2f}s")

        best = max(summary.items(), key=lambda x: x[1]["avg_overall_score"])
        print(f"\n  BEST MODEL: {best[0]} (score: {best[1]['avg_overall_score']:.3f})")
    else:
        print("\n  No summary data in report.")

    print("\n  Output locations (Google Drive):")
    print("  - QLoRA adapter:  MyDrive/arohan_qlora_adapter/")
    print("  - DPO adapter:    MyDrive/arohan_dpo_adapter/")
    print("  - Eval report:    MyDrive/arohan_eval_results/")
    print("=" * 80)
else:
    print("\n  No evaluation report yet.")
    print(f"  Checked: {report_path}")

---
## Pipeline Summary

### What was trained
| Stage | Dataset | Model | Output |
|-------|---------|-------|--------|
| 1. QLoRA (SFT) | 1,040 first aid Q&A pairs | Llama 3 8B + LoRA (r=16) | `arohan_qlora_adapter/` (~50 MB) |
| 2. DPO | 42 preference pairs | QLoRA adapter + DPO alignment | `arohan_dpo_adapter/` (~50 MB) |
| 3. Evaluation | 20 test cases | All models compared | `comprehensive_eval_report.json` |

### If you want to re-run
- Change `ACTIVE_EXPERIMENT` in Cell 5 to try baseline or high config
- Set `GROQ_API_KEY` env var before Cell 7 for Groq comparison
- Delete adapter folders from Drive to force re-training

### Next steps after Colab
1. Copy the best adapter to your local machine
2. Test with: `python inference_qlora.py --adapter ./arohan_qlora_adapter/experiment_medium`
3. Option A: Replace Groq entirely with local Ollama deployment
4. Option B: Use dual mode (Groq for production, local for fallback)